In [ ]:
# Notebook 12: API Deployment

!pip install fastapi uvicorn pydantic chromadb transformers torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.7 MB/s eta 0

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
import json


In [ ]:

# Configuration
class Config:
    MODEL_DIR = BASE_DIR / "models/sft_model/final"  # Changed from ppo
    CHROMA_DIR = BASE_DIR / "data/vector_store"
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    MAX_NEW_TOKENS = 150
    TEMPERATURE = 0.7
    TOP_K = 5

config = Config()
print(f"Using device: {config.DEVICE}")

Using device: cpu


In [ ]:

# Load models
print("Loading models...")

# LLM
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_DIR)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(config.DEVICE)
model.eval()

# Embedding model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Vector store
chroma_client = chromadb.PersistentClient(path=str(config.CHROMA_DIR))
collection = chroma_client.get_collection("arxiv_papers")

print("Models loaded successfully")

Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded successfully


In [ ]:

# API Models
class QueryRequest(BaseModel):
    query: str
    top_k: Optional[int] = 5
    use_rag: Optional[bool] = True

class Source(BaseModel):
    arxiv_id: str
    title: str
    authors: str
    relevance_score: float
    text_snippet: str

class QueryResponse(BaseModel):
    query: str
    response: str
    sources: List[Source]
    metadata: dict

In [ ]:

# RAG functions
def retrieve_context(query: str, top_k: int = 5):
    """Retrieve relevant documents"""
    query_embedding = embedding_model.encode([query])[0].tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    contexts = []
    sources = []

    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        contexts.append(doc)
        sources.append(Source(
            arxiv_id=metadata['arxiv_id'],
            title=metadata['title'],
            authors=metadata['authors'],
            relevance_score=1.0 - distance,  # Convert distance to similarity
            text_snippet=doc[:200]
        ))

    return "\n\n".join(contexts), sources

def generate_response(query: str, context: Optional[str] = None):
    """Generate response with optional RAG context"""
    if context:
        prompt = f"""Context from research papers:
{context}

Question: {query}

Answer based on the context provided:"""
    else:
        prompt = query

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(config.DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.MAX_NEW_TOKENS,
            temperature=config.TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the answer part
    if "Answer based on the context provided:" in response:
        response = response.split("Answer based on the context provided:")[-1].strip()
    elif query in response:
        response = response.replace(query, "").strip()

    return response

In [ ]:

# FastAPI app
app = FastAPI(
    title="Multi-Agent RAG Research Assistant",
    description="RLHF-aligned research assistant with RAG capabilities",
    version="1.0.0"
)

# CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def root():
    return {
        "message": "Multi-Agent RAG Research Assistant API",
        "endpoints": {
            "query": "/query",
            "health": "/health",
            "stats": "/stats"
        }
    }

@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "vector_store_docs": collection.count(),
        "device": config.DEVICE
    }

@app.get("/stats")
async def get_stats():
    return {
        "total_documents": collection.count(),
        "model": "GPT-2 Medium (RLHF-aligned)",
        "vector_store": "ChromaDB",
        "embedding_model": "all-MiniLM-L6-v2"
    }

@app.post("/query", response_model=QueryResponse)
async def query_endpoint(request: QueryRequest):
    """Main query endpoint with RAG"""
    try:
        sources = []
        context = None

        # Retrieve context if RAG is enabled
        if request.use_rag:
            context, sources = retrieve_context(request.query, request.top_k)

        # Generate response
        response = generate_response(request.query, context)

        return QueryResponse(
            query=request.query,
            response=response,
            sources=sources,
            metadata={
                "used_rag": request.use_rag,
                "num_sources": len(sources),
                "model": "gpt2-medium-ppo-aligned"
            }
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("FastAPI app created")

FastAPI app created


In [ ]:

# Save API code to file
api_code = '''from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

# [Include all the code from cells 3-7 above]

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

api_file = BASE_DIR / "api/app.py"
api_file.parent.mkdir(parents=True, exist_ok=True)

with open(api_file, 'w') as f:
    f.write(api_code)

print(f"API code saved to {api_file}")

API code saved to /content/drive/MyDrive/multi_agent_rag_project/api/app.py


In [ ]:

print("\nTesting API endpoints...")

test_request = QueryRequest(
    query="What are transformers in deep learning?",
    top_k=3,
    use_rag=True
)

# Colab already has event loop running, use await
result = await query_endpoint(test_request)

print(f"\nQuery: {result.query}")
print(f"Response: {result.response}")
print(f"\nSources ({len(result.sources)}):")
for i, source in enumerate(result.sources, 1):
    print(f"{i}. {source.title}")
    print(f"   Relevance: {source.relevance_score:.4f}")


Testing API endpoints...

Query: What are transformers in deep learning?
Response: Context from research papers:
Future advancements in this ﬁeld are expected to leverage the Transformer architecture (Vaswani et al., 2017), which has proven to be faster and more e!cient for language modelling compared to LSTMs or CNNs. Although Transformers will be discussed in more detail later, they are brieﬂy introduced here as language representation models. Encoder-decoder structures are common in competitive neuronal sequence transduction models. The model is autoregressive at each phase, using the previous symbols as extra input to construct the next. Transformers’ encoder converts an input series of symbol representations (x1,. . . , xn) into an equivalent sequence of continuous representations, z = (z1, . . . , zn). Then the decoder produces a sequence (y1,. . . , ym) of symbols, starting with z. Following its general architecture, the Trans- former uses layered self-attention and point-wise,

In [ ]:

# Instructions for running
print("\n" + "="*80)
print("API DEPLOYMENT INSTRUCTIONS")
print("="*80)

instructions = """
To run the API server:

1. From terminal:
   cd api
   python app.py

2. Or using uvicorn directly:
   uvicorn app:app --host 0.0.0.0 --port 8000 --reload

3. Access the API:
   - Docs: http://localhost:8000/docs
   - Health: http://localhost:8000/health
   - Query: POST http://localhost:8000/query

4. Example curl request:
   curl -X POST http://localhost:8000/query \\
     -H "Content-Type: application/json" \\
     -d '{"query": "What is RLHF?", "top_k": 5, "use_rag": true}'

5. Deploy to production:
   - Use Docker for containerization
   - Deploy to AWS/GCP/Azure
   - Use Nginx as reverse proxy
   - Enable HTTPS with SSL certificates
"""

print(instructions)

print("\nNotebook 12 Complete!")


API DEPLOYMENT INSTRUCTIONS

To run the API server:

1. From terminal:
   cd api
   python app.py

2. Or using uvicorn directly:
   uvicorn app:app --host 0.0.0.0 --port 8000 --reload

3. Access the API:
   - Docs: http://localhost:8000/docs
   - Health: http://localhost:8000/health
   - Query: POST http://localhost:8000/query

4. Example curl request:
   curl -X POST http://localhost:8000/query \
     -H "Content-Type: application/json" \
     -d '{"query": "What is RLHF?", "top_k": 5, "use_rag": true}'

5. Deploy to production:
   - Use Docker for containerization
   - Deploy to AWS/GCP/Azure
   - Use Nginx as reverse proxy
   - Enable HTTPS with SSL certificates


Notebook 12 Complete!
